# 🤖 ReACT 기반의 Single Agent 구현하기

이번 실습에서는 **ReACT(Reasoning + Acting)** 패턴을 활용하여 사용자의 요구사항을 달성할 때까지 자율적으로 동작하는 에이전트를 구현해봅니다.

> 📢 **ReACT 패턴이란?**
> 
> ReACT는 **Re**asoning(추론)과 **Act**ing(행동)을 결합한 프레임워크입니다.
> - **Reasoning**: LLM이 현재 상황을 분석하고 다음 행동을 계획
> - **Acting**: 계획된 행동(도구 호출)을 실행
> - **Loop**: 목표 달성까지 이 과정을 반복

### 🎯 학습 목표
1. LangGraph의 StateGraph로 ReACT 에이전트 구현하기
2. Tool 정의 및 LLM에 바인딩하는 방법 이해하기
3. 조건부 엣지(Conditional Edge)를 활용한 루프 구현하기

## 📋 목차

1. [환경 설정](#1-환경-설정)
2. [도구(Tool) 정의](#2-도구tool-정의)
3. [상태(State) 정의](#3-상태state-정의)
4. [노드 함수 구현](#4-노드-함수-구현)
5. [그래프 구성 및 컴파일](#5-그래프-구성-및-컴파일)
6. [에이전트 실행 및 테스트](#6-에이전트-실행-및-테스트)

---
## 1. 환경 설정

필요한 패키지를 설치하고 API 키를 설정합니다.

In [ ]:
# 필요한 패키지 설치
%pip install -qU langchain langgraph langchain-openai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# API 키 설정
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = "your-openai-api-key"

print("✅ 환경 설정 완료!")

---
## 2. 도구(Tool) 정의

에이전트가 사용할 도구를 정의합니다. 이 예제에서는 간단한 검색 도구와 계산기 도구를 만들어봅니다.

In [ ]:
from langchain_core.tools import tool

@tool
def search(query: str) -> str:
    """
    주어진 쿼리로 정보를 검색합니다.
    최신 정보나 사실 확인이 필요할 때 사용하세요.
    
    Args:
        query: 검색할 질문이나 키워드
    """
    # 실제로는 검색 API를 호출하겠지만, 여기서는 시뮬레이션
    mock_results = {
        "LangGraph": "LangGraph는 LangChain에서 만든 에이전트 워크플로우 프레임워크입니다. StateGraph를 사용하여 복잡한 에이전트 로직을 구현할 수 있습니다.",
        "ReACT": "ReACT는 Reasoning과 Acting을 결합한 에이전트 패턴으로, LLM이 추론과 행동을 번갈아 수행합니다.",
    }
    for key, value in mock_results.items():
        if key.lower() in query.lower():
            return value
    return f"'{query}'에 대한 검색 결과: 관련 정보를 찾지 못했습니다."

@tool
def calculator(expression: str) -> str:
    """
    수학 표현식을 계산합니다.
    
    Args:
        expression: 계산할 수학 표현식 (예: "2 + 3 * 4")
    """
    try:
        result = eval(expression)
        return f"{expression} = {result}"
    except Exception as e:
        return f"계산 오류: {str(e)}"

# 도구 리스트
tools = [search, calculator]

print("✅ 도구 정의 완료!")
print(f"   - search: 정보 검색")
print(f"   - calculator: 수학 계산")

---
## 3. 상태(State) 정의

LangGraph에서 상태(State)는 그래프 전체에서 공유되는 데이터입니다.

**MessagesState**를 사용하면 메시지 리스트를 자동으로 관리할 수 있습니다.

In [ ]:
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langgraph.graph import MessagesState
import operator

# MessagesState 사용 (messages 필드가 자동으로 누적됨)
# 또는 직접 정의할 수 있음:
class AgentState(TypedDict):
    messages: Annotated[list, operator.add]  # 메시지 누적
    iteration_count: int  # 반복 횟수 추적 (선택적)

print("✅ 상태(State) 정의 완료!")
print("   - messages: 대화 메시지 리스트")
print("   - iteration_count: ReACT 루프 반복 횟수")

---
## 4. 노드 함수 구현

ReACT 에이전트의 핵심 노드들을 구현합니다:
1. **call_model**: LLM을 호출하여 추론 및 도구 호출 결정
2. **should_continue**: 도구 호출 필요 여부 판단 (조건부 라우팅)

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.prebuilt import ToolNode

# LLM 초기화 및 도구 바인딩
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
llm_with_tools = llm.bind_tools(tools)

# 시스템 프롬프트
SYSTEM_PROMPT = """You are a helpful AI assistant with access to tools.
Use the available tools to answer user questions accurately.
Think step by step and use tools when needed.
Always respond in Korean."""

def call_model(state: MessagesState):
    """LLM을 호출하여 응답 또는 도구 호출 결정"""
    print("---🧠 REASONING (모델 호출)---")
    
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    response = llm_with_tools.invoke(messages)
    
    return {"messages": [response]}

def should_continue(state: MessagesState) -> Literal["tools", "__end__"]:
    """도구 호출 필요 여부 판단"""
    last_message = state["messages"][-1]
    
    # tool_calls가 있으면 도구 실행
    if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
        print("---🔧 ACTING (도구 호출 필요)---")
        return "tools"
    
    # 없으면 종료
    print("---✅ COMPLETE (응답 완료)---")
    return "__end__"

# ToolNode 생성 (도구 실행 담당)
tool_node = ToolNode(tools)

print("✅ 노드 함수 구현 완료!")

---
## 5. 그래프 구성 및 컴파일

ReACT 에이전트의 그래프 구조:
```
START → call_model → [tool_calls 있음?]
                       ├─ Yes → tools → call_model (루프)
                       └─ No → END
```

In [ ]:
from langgraph.graph import StateGraph, START, END

# 그래프 생성
workflow = StateGraph(MessagesState)

# 노드 추가
workflow.add_node("call_model", call_model)
workflow.add_node("tools", tool_node)

# 엣지 연결
workflow.add_edge(START, "call_model")

# 조건부 엣지: call_model 후 도구 호출 여부에 따라 분기
workflow.add_conditional_edges(
    "call_model",
    should_continue,
    {
        "tools": "tools",
        "__end__": END
    }
)

# 도구 실행 후 다시 모델 호출 (ReACT 루프)
workflow.add_edge("tools", "call_model")

# 그래프 컴파일
react_agent = workflow.compile()

print("✅ ReACT 에이전트 컴파일 완료!")

In [ ]:
# 그래프 시각화
from IPython.display import Image, display

try:
    display(Image(react_agent.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"시각화 오류: {e}")
    print("\n그래프 구조:")
    print("START → call_model → [tools/END] → call_model → END")

---
## 6. 에이전트 실행 및 테스트

다양한 질문으로 ReACT 에이전트를 테스트해봅니다.

In [ ]:
# 테스트 1: 일반 대화 (도구 호출 없음)
test1 = "안녕하세요! 반갑습니다."

print(f"💬 질문: {test1}")
print("=" * 50)

result = react_agent.invoke({"messages": [{"role": "user", "content": test1}]})
print(f"\n🤖 답변:\n{result['messages'][-1].content}")

In [ ]:
# 테스트 2: 검색 도구 호출
test2 = "LangGraph가 뭔지 검색해서 알려줘"

print(f"💬 질문: {test2}")
print("=" * 50)

result = react_agent.invoke({"messages": [{"role": "user", "content": test2}]})
print(f"\n🤖 답변:\n{result['messages'][-1].content}")

In [ ]:
# 테스트 3: 계산 도구 호출
test3 = "(15 + 27) * 3을 계산해줘"

print(f"💬 질문: {test3}")
print("=" * 50)

result = react_agent.invoke({"messages": [{"role": "user", "content": test3}]})
print(f"\n🤖 답변:\n{result['messages'][-1].content}")

In [ ]:
# 테스트 4: 스트리밍으로 실행 과정 확인
test4 = "ReACT 패턴에 대해 검색하고 설명해줘"

print(f"💬 질문: {test4}")
print("=" * 50)
print("\n🔄 실행 과정:")

for chunk in react_agent.stream({"messages": [{"role": "user", "content": test4}]}):
    for node_name, value in chunk.items():
        print(f"\n📍 [{node_name}] 노드 실행")
        
        if "messages" in value and value["messages"]:
            last_msg = value["messages"][-1]
            
            if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
                for tc in last_msg.tool_calls:
                    print(f"   🔧 Tool: {tc['name']}({tc['args']})")
            elif hasattr(last_msg, 'content') and last_msg.content:
                if node_name == "tools":
                    print(f"   📋 결과: {last_msg.content[:100]}...")
                else:
                    print(f"\n🤖 최종 답변:\n{last_msg.content}")

---
## 💡 정리

### ReACT 에이전트의 핵심 구성요소

| 구성요소 | 역할 |
|---------|------|
| **StateGraph** | 에이전트의 상태와 흐름을 관리하는 그래프 |
| **State** | 그래프 전체에서 공유되는 데이터 (messages 등) |
| **call_model** | LLM을 호출하여 추론 및 도구 호출 결정 |
| **ToolNode** | 도구 실행을 담당하는 prebuilt 노드 |
| **add_conditional_edges** | 조건에 따라 다음 노드를 결정하는 분기 |

### ReACT 패턴의 장점
- ✅ **자율적 실행**: 목표 달성까지 자동으로 반복
- ✅ **도구 활용**: 외부 도구를 통해 능력 확장
- ✅ **추론 가능**: 각 단계에서 왜 그런 결정을 했는지 추적 가능

### 다음 단계
👉 **CH02._02.** Prompt Chaining으로 더 똑똑한 답변 받기